# 跨境电商销量预测 - Chronos-2 SageMaker 部署

基于 Amazon Chronos-2 预训练时序模型，支持协变量（节假日、促销、广告等）

## 1. 安装依赖

In [ ]:
!pip install -U -q "sagemaker<3" pandas matplotlib

In [ ]:
import sys
sys.path.append("code_preprocess")

import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime, timedelta
from preprocess import preprocess_data, PAST_COVARIATES, FUTURE_COVARIATES

## 2. 配置参数

In [ ]:
ROLE = None  # SageMaker Notebook 中可设为 None
PREDICTION_LENGTH = 28
FREQ = "D"
MARKETPLACE = "US"  # US / UK / DE / FR / IT / ES / CA

## 3. 加载并预处理数据

In [ ]:
# ========== 替换为真实数据 ==========
np.random.seed(42)
dates = pd.date_range("2024-01-01", "2024-12-31", freq="D")
asins = ["B0ASIN001", "B0ASIN002", "B0ASIN003"]

data = []
for asin in asins:
    base_sales = np.random.randint(50, 200)
    for date in dates:
        seasonal = 1.0
        if date.month == 10:
            seasonal = 2.5
        elif date.month in [11, 12]:
            seasonal = 3.0
        weekend = 1.2 if date.dayofweek >= 5 else 1.0
        sales = int(base_sales * seasonal * weekend * np.random.uniform(0.8, 1.2))
        
        data.append({
            "asin": asin,
            "date": date,
            "sales_quantity": sales,
            "sale_price": np.random.uniform(20, 50),
            "original_price": 50,
            "ad_spend": np.random.uniform(50, 200),
            "ad_impressions": np.random.randint(5000, 20000),
            "ad_clicks": np.random.randint(100, 500),
            "fba_inventory": np.random.randint(500, 2000),
            "is_lightning_deal": 1 if np.random.random() < 0.05 else 0,
            "is_coupon_active": 1 if np.random.random() < 0.1 else 0,
            "launch_date": "2023-06-01",
        })

raw_df = pd.DataFrame(data)
print(f"数据量: {len(raw_df)} 行, {raw_df['asin'].nunique()} 个产品")
raw_df.head()

In [ ]:
df = preprocess_data(raw_df, marketplace=MARKETPLACE)
print(f"特征数量: {len(df.columns)}")
df.head()

## 4. 部署 Chronos-2 端点

In [ ]:
from sagemaker.jumpstart.model import JumpStartModel

js_model = JumpStartModel(
    model_id="pytorch-forecasting-chronos-2",
    instance_type="ml.g5.xlarge",
    role=ROLE,
)
predictor = js_model.deploy()
print("端点部署完成")

## 5. 构建预测请求

In [ ]:
def build_chronos_payload(df, target_col="sales_quantity", id_col="asin", 
                          date_col="date", prediction_length=28, freq="D",
                          past_cov_cols=None, future_cov_cols=None):
    """构建 Chronos-2 API 请求"""
    inputs = []
    
    for item_id, group in df.sort_values([id_col, date_col]).groupby(id_col):
        entry = {
            "target": group[target_col].tolist(),
            "item_id": str(item_id),
            "start": group[date_col].iloc[0].isoformat(),
        }
        
        if past_cov_cols:
            valid_cols = [c for c in past_cov_cols if c in group.columns]
            entry["past_covariates"] = {
                col: group[col].fillna(0).tolist() for col in valid_cols
            }
        
        if future_cov_cols:
            last_date = group[date_col].max()
            future_dates = pd.date_range(last_date + timedelta(days=1), 
                                         periods=prediction_length, freq=freq)
            future_df = pd.DataFrame({date_col: future_dates})
            future_df = preprocess_data(
                future_df.assign(asin=item_id, sales_quantity=0), 
                marketplace=MARKETPLACE
            )
            valid_cols = [c for c in future_cov_cols if c in future_df.columns]
            entry["future_covariates"] = {
                col: future_df[col].fillna(0).tolist() for col in valid_cols
            }
        
        inputs.append(entry)
    
    return {
        "inputs": inputs,
        "parameters": {
            "prediction_length": prediction_length,
            "freq": freq,
            "quantile_levels": [0.1, 0.5, 0.9]
        }
    }

In [ ]:
payload = build_chronos_payload(
    df,
    prediction_length=PREDICTION_LENGTH,
    freq=FREQ,
    past_cov_cols=PAST_COVARIATES,
    future_cov_cols=FUTURE_COVARIATES,
)
print(f"预测 {len(payload['inputs'])} 个产品, 每个预测 {PREDICTION_LENGTH} 天")

## 6. 执行预测

In [ ]:
response = predictor.predict(payload)

def response_to_df(response, freq="D"):
    dfs = []
    for pred in response["predictions"]:
        forecast_df = pd.DataFrame({
            "asin": pred.get("item_id"),
            "date": pd.date_range(pred["start"], periods=len(pred["mean"]), freq=freq),
            "forecast": pred["mean"],
            "lower_10": pred["0.1"],
            "upper_90": pred["0.9"],
        })
        dfs.append(forecast_df)
    return pd.concat(dfs, ignore_index=True)

forecast_df = response_to_df(response, freq=FREQ)
forecast_df

## 7. 可视化

In [ ]:
def plot_forecast(df, forecast_df, asin, history_days=60):
    hist = df[df["asin"] == asin].tail(history_days).set_index("date")["sales_quantity"]
    pred = forecast_df[forecast_df["asin"] == asin].set_index("date")
    
    plt.figure(figsize=(14, 5))
    plt.plot(hist.index, hist.values, label="历史销量", color="#2E86AB", linewidth=2)
    plt.plot(pred.index, pred["forecast"], label="预测销量", color="#E94F37", linewidth=2)
    plt.fill_between(pred.index, pred["lower_10"], pred["upper_90"], 
                     alpha=0.3, color="#E94F37", label="90% 置信区间")
    
    plt.title(f"产品 {asin} 销量预测", fontsize=14)
    plt.xlabel("日期")
    plt.ylabel("销量")
    plt.legend(loc="upper left")
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

for asin in forecast_df["asin"].unique():
    plot_forecast(df, forecast_df, asin)

## 8. 导出结果

In [ ]:
output_path = f"forecast_{MARKETPLACE}_{datetime.now().strftime('%Y%m%d')}.csv"
forecast_df.to_csv(output_path, index=False)
print(f"预测结果已保存: {output_path}")

## 9. 清理资源

In [ ]:
# predictor.delete_predictor()